# Drift Evaluation Completo (Sintético + Estado + Episodios)

Notebook listo para **descargar y ejecutar** que implementa:

- Carga de datos sintéticos y etiquetado manual (`etiquetado_manual.csv`).
- Uso directo de tus funciones reales desde `../Analisis/Funciones_Drift.py`.
- Detección de drift **con estado por variable** (`NORMAL` / `EN_DRIFT`).
- **Referencia congelada** al detectar el primer drift.
- Soporte para "drift sobre drift" (episodios anidados, marcados con `episode_id` distinto).
- Construcción de **episodios automáticos** a partir de ventanas deslizantes.
- Comparación **episodio vs episodio** contra el etiquetado manual.
- Cálculo de métricas de detección a nivel de episodio (Precision, Recall, F1).
- Runner multi-ventana (por defecto: `6H`, `12H`, `24H`, `48H`).  
- Gráficos 5x2 (`plot_grid_episodes_5x2`) con bandas rojas (manual) y azules (auto).
- Exportación de CSVs con ventanas, episodios y evaluación.

Supone que tienes este notebook en la carpeta `Drift Evaluation` junto a:
- `../Analisis/Funciones_Drift.py`
- `synthetic_data/synthetic_plant.csv`
- `synthetic_data/etiquetado_manual.csv`


## 1. Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import importlib.util

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import plotly.express as px

## 2. Cargar módulo `Funciones_Drift.py` y alias útiles

In [2]:
spec = importlib.util.spec_from_file_location("funciones_drift", "../Analisis/Funciones_Drift.py")
funciones_drift = importlib.util.module_from_spec(spec)
spec.loader.exec_module(funciones_drift)

# Aliases a funciones que usaremos
strip_outliers         = funciones_drift.strip_outliers
ref_decay_prefix_mass  = funciones_drift.ref_decay_prefix_mass
ref_golden             = funciones_drift.ref_golden
ref_seasonal           = funciones_drift.ref_seasonal
psi_numeric            = funciones_drift.psi_numeric
score_numeric_series   = funciones_drift._score_numeric_series

print("✔ Funciones_Drift.py cargado y aliases creados")

✔ Funciones_Drift.py cargado y aliases creados


## 3. Carga de serie sintética y construcción de intervalos manuales

In [3]:
# Paths de entrada
SERIES_PATH = Path("synthetic_data/synthetic_plant.csv")
LABELS_PATH = Path("synthetic_data/synthetic_plant_events.csv")

assert SERIES_PATH.exists(), f"No se encontró {SERIES_PATH}"
assert LABELS_PATH.exists(),  f"No se encontró {LABELS_PATH}"

# --- Serie sintética ---
df_raw = pd.read_csv(SERIES_PATH)
# Detectar columna de tiempo si no se llama exactamente 'date_time'
if "date_time" not in df_raw.columns:
    for c in ["datetime","timestamp","time","fecha","tiempo"]:
        if c in df_raw.columns:
            df_raw = df_raw.rename(columns={c:"date_time"})
            break

df_raw["date_time"] = pd.to_datetime(df_raw["date_time"], errors="coerce")
df_raw = (df_raw
          .dropna(subset=["date_time"])
          .sort_values("date_time")
          .set_index("date_time"))

# Quitamos outliers usando tu helper
df_raw = strip_outliers(df_raw)

# Nos quedamos con columnas numéricas (las típicas var_1, var_2, etc.)
df = df_raw.select_dtypes(include="number").copy()
assert not df.empty, "No hay columnas numéricas en la serie sintética."

t_min, t_max = df.index.min(), df.index.max()

# --- Etiquetado manual ---
events = pd.read_csv(LABELS_PATH)
events["date_time"] = pd.to_datetime(events["date_time"], errors="coerce")
events = (events
          .dropna(subset=["date_time","variable","event"])
          .assign(event=lambda s: s["event"].str.lower().str.strip())
          .query("event in ['start','end']")
          .sort_values(["variable","date_time"])
          .reset_index(drop=True))

def events_to_intervals(ev: pd.DataFrame) -> pd.DataFrame:
    """
    Convierte pares start/end por variable en episodios continuos (manuales).
    Si existe columna 'drift_type' en el CSV de eventos, la conserva por episodio.
    Se asume que start y end de un mismo episodio tienen el mismo drift_type.
    """
    has_type = "drift_type" in ev.columns
    rows = []
    for var, g in ev.groupby("variable", sort=True):
        open_t = None
        open_type = None
        for _, r in g.iterrows():
            evt = str(r["event"]).lower()
            dt_val = r["drift_type"] if has_type else "unknown"

            if evt == "start":
                open_t = r["date_time"]
                open_type = dt_val
            elif evt == "end" and open_t is not None and r["date_time"] > open_t:
                rows.append({
                    "variable": var,
                    "manual_start": open_t,
                    "manual_end": r["date_time"],
                    "drift_type": open_type,
                })
                open_t = None
                open_type = None
    return pd.DataFrame(rows)

intervals_manual = events_to_intervals(events)

## 4. Motor stateful de detección de drift (PSI + referencia congelada + episodios)

In [4]:
def run_drift_for_strategy_multi_metric(
    df: pd.DataFrame,
    window: str,
    strategy: str,
    metrics: tuple = ("psi", "ks", "wasserstein", "mannwhitney"),
    thresholds: dict | None = None,
    min_points: int = 5,
):
    """
    Ejecuta detección de drift para una estrategia dada y *varias* métricas numéricas.

    - NO hay referencia congelada.
    - En cada ventana se construye la referencia según `strategy` usando TODO el historial hasta t0.
    - Para cada métrica en `metrics`:
        stat_value = score_numeric_series(ref, cur, metric)
        drift_flag = stat_value >= threshold_métrica
    - Un episodio es una secuencia contigua de ventanas con drift_flag=True.
    - Cuando la métrica baja del umbral, se cierra el episodio y listo; la referencia
      en pasos futuros ya incorpora el nuevo nivel estable.

    `thresholds`: dict opcional {metric_name: valor}. Si no se pasa, usa los defaults
    coherentes con tu Funciones_Drift:

        psi         ~ 0.2
        ks          ~ 0.15
        wasserstein -> 0.5 * std(ref) si no se fija
        mannwhitney ~ 0.55
    """

    # Defaults coherentes con _dispatch_build_metrics
    default_thr = {"psi": 0.2, "ks": 0.15, "wasserstein": np.nan, "mannwhitney": 0.55}
    thresholds = thresholds or {}

    w = pd.to_timedelta(window)
    t_min, t_max = df.index.min(), df.index.max()
    t_ends = pd.date_range(t_min + w, t_max, freq=window)

    variables = list(df.columns)

    # Estado por (métrica, variable) sólo para numerar episodios contiguos
    state = {
        metric: {var: "NORMAL" for var in variables}
        for metric in metrics
    }
    current_episode = {
        metric: {var: 0 for var in variables}
        for metric in metrics
    }

    rows = []

    for t_end in t_ends:
        t0 = t_end - w

        df_hist = df.loc[: t0 - pd.Timedelta(microseconds=1)]
        df_cur  = df.loc[t0:t_end]

        if df_hist.empty or df_cur.empty:
            continue

        # Referencia según la estrategia (SIEMPRE recalculada)
        if strategy == "decay":
            ref_global = ref_decay_prefix_mass(df_hist, now=t_end)
        elif strategy == "golden":
            ref_global = ref_golden(df_hist)
        elif strategy == "seasonal":
            ref_global = ref_seasonal(df_hist, current_end=t_end)
        else:
            raise ValueError(f"Estrategia desconocida: {strategy}")

        if ref_global is None or ref_global.empty:
            ref_global = df_hist

        for var in variables:
            cur_series = df_cur[var].dropna()

            if cur_series.size < min_points:
                # Sin datos suficientes: logueamos filas sin drift (no cambiamos estado)
                for metric_name in metrics:
                    rows.append({
                        "variable": var,
                        "strategy": strategy,
                        "window": window,
                        "metric": metric_name,
                        "t0": t0,
                        "t1": t_end,
                        "drift_flag": False,
                        "episode_id": np.nan,
                        "stat_value": None,
                        "threshold": None,
                        "state": state[metric_name][var],
                    })
                continue

            # Referencia para esta variable
            if var in ref_global.columns:
                ref_series = ref_global[var].dropna()
            else:
                ref_series = df_hist[var].dropna()

            for metric_name in metrics:
                base_thr = default_thr.get(metric_name, 0.2)
                thr = thresholds.get(metric_name, base_thr)

                if ref_series.empty:
                    stat_val = None
                    eff_thr = thr
                    drift_flag = False
                else:
                    # Usa exactamente la misma función que el pipeline
                    stat_val = score_numeric_series(ref_series, cur_series, metric_name)

                    # Para Wasserstein, si el threshold viene como NaN, lo adaptamos a la escala ref
                    eff_thr = thr
                    if metric_name == "wasserstein" and (eff_thr is None or (isinstance(eff_thr, float) and np.isnan(eff_thr))):
                        std_ref = pd.to_numeric(ref_series, errors="coerce").dropna().std()
                        eff_thr = float(std_ref) * 0.5 if pd.notna(std_ref) else 0.5

                    if stat_val is None or np.isnan(stat_val):
                        drift_flag = False
                    else:
                        drift_flag = bool(stat_val >= eff_thr)

                # Actualizar estado/episodios (sin referencia congelada)
                if drift_flag:
                    if state[metric_name][var] == "NORMAL":
                        current_episode[metric_name][var] += 1
                        state[metric_name][var] = "DRIFT"
                else:
                    if state[metric_name][var] == "DRIFT":
                        state[metric_name][var] = "NORMAL"

                rows.append({
                    "variable": var,
                    "strategy": strategy,
                    "window": window,
                    "metric": metric_name,
                    "t0": t0,
                    "t1": t_end,
                    "drift_flag": drift_flag,
                    "episode_id": (
                        current_episode[metric_name][var]
                        if drift_flag else np.nan
                    ),
                    "stat_value": stat_val,
                    "threshold": eff_thr,
                    "state": state[metric_name][var],
                })

    return pd.DataFrame(rows)


def run_drift_all_multi_metric(
    df: pd.DataFrame,
    windows=("6H","12H","24H","48H"),
    strategies=("decay","golden","seasonal"),
    metrics: tuple = ("psi", "ks", "wasserstein", "mannwhitney"),
    thresholds: dict | None = None,
    min_points: int = 5,
):
    """Runner multi-ventana, multi-estrategia y multi-métrica."""
    all_frames = []
    for win in windows:
        for strat in strategies:
            print(f"[DRIFT] window={win}, strategy={strat}")
            dfw = run_drift_for_strategy_multi_metric(
                df=df,
                window=win,
                strategy=strat,
                metrics=metrics,
                thresholds=thresholds,
                min_points=min_points,
            )
            all_frames.append(dfw)

    if not all_frames:
        return pd.DataFrame()
    return pd.concat(all_frames, ignore_index=True)

In [5]:
EVAL_WINDOWS = ["4H", "6H", "12H", "24H", "48H"]
STRATEGIES   = ["decay","golden","seasonal"]
METRICS      = ("psi", "ks", "wasserstein", "mannwhitney")

# Si quieres tuneo fino de thresholds por métrica, lo puedes hacer aquí
METRIC_THRESHOLDS = {
    # si no defines alguno, se usan los defaults del motor (psi~0.2, ks~0.15, mw~0.55)
    # "psi": 0.2,
    # "ks": 0.15,
    # "wasserstein": np.nan,   # → se adaptará a 0.5 * std(ref)
    # "mannwhitney": 0.55,
}

df_windows = run_drift_all_multi_metric(
    df=df,
    windows=EVAL_WINDOWS,
    strategies=STRATEGIES,
    metrics=METRICS,
    thresholds=METRIC_THRESHOLDS,
    min_points=5,
)

print("Shape df_windows:", df_windows.shape)
df_windows.head()


[DRIFT] window=4H, strategy=decay
[DRIFT] window=4H, strategy=golden
[DRIFT] window=4H, strategy=seasonal
[DRIFT] window=6H, strategy=decay
[DRIFT] window=6H, strategy=golden
[DRIFT] window=6H, strategy=seasonal
[DRIFT] window=12H, strategy=decay
[DRIFT] window=12H, strategy=golden
[DRIFT] window=12H, strategy=seasonal
[DRIFT] window=24H, strategy=decay
[DRIFT] window=24H, strategy=golden
[DRIFT] window=24H, strategy=seasonal
[DRIFT] window=48H, strategy=decay
[DRIFT] window=48H, strategy=golden
[DRIFT] window=48H, strategy=seasonal
Shape df_windows: (31200, 11)


,variable,strategy,window,metric,t0,t1,drift_flag,episode_id,stat_value,threshold,state
0,var_1,decay,4H,psi,2025-01-01 04:00:00,2025-01-01 08:00:00,False,NaN,0.163636,0.200000,NORMAL
1,var_1,decay,4H,ks,2025-01-01 04:00:00,2025-01-01 08:00:00,True,1.0,0.160297,0.150000,DRIFT
2,var_1,decay,4H,wasserstein,2025-01-01 04:00:00,2025-01-01 08:00:00,False,NaN,0.154868,0.201487,NORMAL
3,var_1,decay,4H,mannwhitney,2025-01-01 04:00:00,2025-01-01 08:00:00,True,1.0,0.576181,0.550000,DRIFT
4,var_2,decay,4H,psi,2025-01-01 04:00:00,2025-01-01 08:00:00,False,NaN,0.074846,0.200000,NORMAL


## 5. Compactar ventanas en episodios automáticos

In [6]:
def windows_to_episodes_multi_metric(df_windows: pd.DataFrame) -> pd.DataFrame:
    """
    Compacta secuencias de ventanas con drift_flag=1
    en episodios automáticos por (window, strategy, metric, variable, episode_id).
    """
    dfw = df_windows.copy()
    dfw = dfw[dfw["drift_flag"] == True].dropna(subset=["episode_id"])
    if dfw.empty:
        return pd.DataFrame(columns=[
            "window","strategy","metric","variable","episode_id",
            "seg_start","seg_end","seg_length","stat_max"
        ])

    rows = []
    for keys, sub in dfw.groupby(
        ["window","strategy","metric","variable","episode_id"],
        dropna=False
    ):
        win, strat, metric, var, eid = keys
        sub = sub.sort_values("t0")
        seg_start = sub["t0"].min()
        seg_end   = sub["t1"].max()
        stat_max  = sub["stat_value"].max()
        rows.append({
            "window": win,
            "strategy": strat,
            "metric": metric,
            "variable": var,
            "episode_id": int(eid),
            "seg_start": seg_start,
            "seg_end": seg_end,
            "seg_length": seg_end - seg_start,
            "stat_max": stat_max,
        })
    return pd.DataFrame(rows)


df_episodes_auto = windows_to_episodes_multi_metric(df_windows)
print("Episodios automáticos:", len(df_episodes_auto))

Episodios automáticos: 2685


## 6. Evaluación episodio-a-episodio vs etiquetado manual

In [9]:
def evaluate_episodes_vs_manual_multi_metric(
    df_episodes_auto: pd.DataFrame,
    intervals_manual: pd.DataFrame,
):
    """
    Compara episodios automáticos vs episodios manuales por (window, strategy, metric):

      Para cada (window, strategy, metric):
        - TP: episodios manuales que tienen al menos un solape con algún episodio auto.
        - FN: episodios manuales sin solape.
        - FP: episodios auto sin solape con ningún episodio manual.

    Además calcula:
        - coverage_mean / coverage_median: fracción de cada episodio manual que queda cubierto.
        - delay_mean_hours / delay_median_hours: retraso entre manual_start y primera detección.
        - false_alarms_per_day: FP normalizados por la duración total de la serie.
    """
    if df_episodes_auto.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    man = intervals_manual.copy()

    # Duración total de la serie (para tasas por día)
    total_days = max((t_max - t_min).total_seconds() / (3600 * 24), 1e-9)

    results = []
    marks_manual_all = []
    marks_auto_all   = []

    for (win, strat, metric), auto_sub in df_episodes_auto.groupby(
        ["window","strategy","metric"], dropna=False
    ):
        vars_in_auto = sorted(auto_sub["variable"].unique())
        man_sub = man[man["variable"].isin(vars_in_auto)].copy()
        if man_sub.empty and auto_sub.empty:
            continue

        # --- Manual: vista TP/FN + coverage + delay ---
        manual_matches = []
        coverage_vals  = []
        delay_vals     = []

        for _, mrow in man_sub.iterrows():
            v  = mrow["variable"]
            ms = mrow["manual_start"]
            me = mrow["manual_end"]

            rel_auto = auto_sub[auto_sub["variable"] == v]

            total_overlap = pd.Timedelta(0)
            first_det = None

            for _, arow in rel_auto.iterrows():
                as_ = arow["seg_start"]
                ae  = arow["seg_end"]

                # solape con el episodio manual
                start = max(ms, as_)
                end   = min(me, ae)
                if end > start:
                    total_overlap += (end - start)

                    # tiempo de detección (clamp en el inicio manual)
                    det_candidate = as_
                    if det_candidate < ms:
                        det_candidate = ms
                    if first_det is None or det_candidate < first_det:
                        first_det = det_candidate

            # ¿Hubo al menos un solape?
            matched = total_overlap > pd.Timedelta(0)
            manual_matches.append(bool(matched))

            # coverage
            dur = me - ms
            if dur.total_seconds() > 0:
                cov_val = total_overlap.total_seconds() / dur.total_seconds()
            else:
                cov_val = 0.0
            coverage_vals.append(cov_val)

            # delay en horas (solo si hubo detección)
            if first_det is None:
                delay_vals.append(np.nan)
            else:
                delay = (first_det - ms).total_seconds() / 3600.0
                if delay < 0:
                    delay = 0.0
                delay_vals.append(delay)

        man_sub["matched_auto"] = manual_matches
        man_sub["coverage"]     = coverage_vals
        man_sub["delay_hours"]  = delay_vals

        TP = int(man_sub["matched_auto"].sum()) if not man_sub.empty else 0
        FN = int((~man_sub["matched_auto"]).sum()) if not man_sub.empty else 0

        # --- Auto: vista FP ---
        auto_sub = auto_sub.copy()
        auto_matches = []
        for _, arow in auto_sub.iterrows():
            v  = arow["variable"]
            as_ = arow["seg_start"]
            ae  = arow["seg_end"]

            overlap = (
                (man_sub["variable"] == v) &
                ~(man_sub["manual_end"] < as_) &
                ~(man_sub["manual_start"] > ae)
            ).any()
            auto_matches.append(overlap)

        auto_sub["matched_manual"] = auto_matches
        FP = int((~auto_sub["matched_manual"]).sum()) if not auto_sub.empty else 0

        prec = TP / (TP + FP) if (TP + FP) > 0 else np.nan
        rec  = TP / (TP + FN) if (TP + FN) > 0 else np.nan
        f1   = (2 * prec * rec) / (prec + rec) if prec and rec and (prec + rec) > 0 else np.nan

        # coverage y delay agregados
        coverage_mean   = float(np.nanmean(coverage_vals))  if coverage_vals else np.nan
        coverage_median = float(np.nanmedian(coverage_vals)) if coverage_vals else np.nan

        delay_valid = [d for d in delay_vals if not np.isnan(d)]
        delay_mean_hours   = float(np.mean(delay_valid))   if delay_valid else np.nan
        delay_median_hours = float(np.median(delay_valid)) if delay_valid else np.nan

        false_alarms_per_day = FP / total_days

        results.append({
            "window": win,
            "strategy": strat,
            "metric": metric,
            "TP_episodes": TP,
            "FP_episodes": FP,
            "FN_episodes": FN,
            "Precision": prec,
            "Recall": rec,
            "F1": f1,
            "coverage_mean": coverage_mean,
            "coverage_median": coverage_median,
            "delay_mean_hours": delay_mean_hours,
            "delay_median_hours": delay_median_hours,
            "false_alarms_per_day": false_alarms_per_day,
        })

        # Guardamos manual/auto marcados (para análisis por tipo de drift, etc.)
        man_sub["window"]   = win
        man_sub["strategy"] = strat
        man_sub["metric"]   = metric

        auto_sub["window"]   = win
        auto_sub["strategy"] = strat
        auto_sub["metric"]   = metric

        marks_manual_all.append(man_sub)
        marks_auto_all.append(auto_sub)

    eval_df = pd.DataFrame(results)
    manual_marked = pd.concat(marks_manual_all, ignore_index=True) if marks_manual_all else pd.DataFrame()
    auto_marked   = pd.concat(marks_auto_all,   ignore_index=True) if marks_auto_all   else pd.DataFrame()

    return eval_df, manual_marked, auto_marked

eval_episodes_df, manual_marked, auto_marked = evaluate_episodes_vs_manual_multi_metric(
    df_episodes_auto=df_episodes_auto,
    intervals_manual=intervals_manual,
)

print("=== Métricas por (window, strategy, metric) ===")
eval_episodes_df

=== Métricas por (window, strategy, metric) ===


,window,strategy,metric,TP_episodes,FP_episodes,FN_episodes,Precision,Recall,F1,coverage_mean,coverage_median,delay_mean_hours,delay_median_hours,false_alarms_per_day
0,12H,decay,ks,21,3,1,0.875000,0.954545,0.913043,0.932722,1.000000,0.950000,0.000000,0.150005
1,12H,decay,mannwhitney,0,14,17,0.000000,0.000000,NaN,0.000000,0.000000,NaN,NaN,0.700024
2,12H,decay,psi,21,2,1,0.913043,0.954545,0.933333,0.926206,1.000000,1.521429,0.000000,0.100003
3,12H,decay,wasserstein,21,1,1,0.954545,0.954545,0.954545,0.926206,1.000000,1.521429,0.000000,0.050002
4,12H,golden,ks,22,24,0,0.478261,1.000000,0.647059,0.943905,1.000000,1.481061,0.000000,1.200042
5,12H,golden,mannwhitney,0,31,17,0.000000,0.000000,NaN,0.000000,0.000000,NaN,NaN,1.550054
6,12H,golden,psi,20,11,2,0.645161,0.909091,0.754717,0.821657,0.990537,2.645833,0.000000,0.550019
7,12H,golden,wasserstein,21,5,1,0.807692,0.954545,0.875000,0.847179,0.990537,3.527778,0.000000,0.250009
8,12H,seasonal,ks,21,21,1,0.500000,0.954545,0.656250,0.933297,1.000000,1.161905,0.000000,1.050036
9,12H,seasonal,mannwhitney,1,34,21,0.028571,0.045455,0.035088,0.005311,0.000000,0.000000,0.000000,1.700059


## 7. Exportar CSVs de resultados

In [10]:
OUTPUT_DIR = Path("synthetic_data/results")
OUTPUT_DIR.mkdir(exist_ok=True)

windows_path  = OUTPUT_DIR / "df_windows_stateful.csv"
episodes_path = OUTPUT_DIR / "df_episodes_auto.csv"
eval_path     = OUTPUT_DIR / "eval_episodes_by_window_strategy.csv"
manual_path   = OUTPUT_DIR / "manual_marked_episodes.csv"
auto_path     = OUTPUT_DIR / "auto_marked_episodes.csv"

df_windows.to_csv(windows_path, index=False)
df_episodes_auto.to_csv(episodes_path, index=False)
eval_episodes_df.to_csv(eval_path, index=False)
if not manual_marked.empty:
    manual_marked.to_csv(manual_path, index=False)
if not auto_marked.empty:
    auto_marked.to_csv(auto_path, index=False)

print("Guardados:")
print(" -", windows_path)
print(" -", episodes_path)
print(" -", eval_path)
if manual_marked is not None and not manual_marked.empty:
    print(" -", manual_path)
if auto_marked is not None and not auto_marked.empty:
    print(" -", auto_path)

Guardados:
 - synthetic_data\results\df_windows_stateful.csv
 - synthetic_data\results\df_episodes_auto.csv
 - synthetic_data\results\eval_episodes_by_window_strategy.csv
 - synthetic_data\results\manual_marked_episodes.csv
 - synthetic_data\results\auto_marked_episodes.csv


In [11]:
print("Métricas por (window, strategy, metric):")
display(eval_episodes_df)

# ============================
# 1) Resúmenes tabulares
# ============================

# Auxiliar: número de horas como entero para ordenar "2H", "12H", "120H", etc.
eval_episodes_df["window_num"] = eval_episodes_df["window"].str.rstrip("H").astype(int)

# Promedio por tamaño de ventana y métrica (macro sobre estrategias)
summary_by_window_metric = (
    eval_episodes_df
    .groupby(["metric", "window", "window_num"])[["Precision", "Recall", "F1"]]
    .mean()
    .reset_index()
    .sort_values(["metric", "window_num"])
    .drop(columns="window_num")
    .set_index(["metric", "window"])
)
print("\n=== Promedio por ventana y métrica ===")
display(summary_by_window_metric.style.format("{:.3f}"))

# Promedio por estrategia y métrica (macro sobre ventanas)
summary_by_strategy_metric = (
    eval_episodes_df
    .groupby(["metric", "strategy"])[["Precision", "Recall", "F1"]]
    .mean()
    .reset_index()
    .sort_values(["metric", "F1"], ascending=[True, False])
    .set_index(["metric", "strategy"])
)
print("\n=== Promedio por estrategia y métrica ===")
display(summary_by_strategy_metric.style.format("{:.3f}"))

# Ranking fino por (métrica, ventana, estrategia)
summary_full = (
    eval_episodes_df
    .groupby(["metric", "window", "window_num", "strategy"])[["Precision", "Recall", "F1"]]
    .mean()
    .reset_index()
    .sort_values(["metric", "window_num", "F1"], ascending=[True, True, False])
    .drop(columns="window_num")
)
print("\n=== Promedio por (métrica, ventana, estrategia) ordenado por F1 ===")
display(
    summary_full.style.format(
        "{:.3f}",
        subset=["Precision", "Recall", "F1"]
    )
)

# ============================
# 2) Heatmaps por métrica
# ============================
if not eval_episodes_df.empty:
    # Orden correcto de ventanas por horas (2H, 4H, 6H, 12H, 24H, 120H, ...)
    window_order = sorted(
        eval_episodes_df["window"].unique(),
        key=lambda w: int(str(w).rstrip("H"))
    )

    for metric_name in eval_episodes_df["metric"].unique():
        sub = eval_episodes_df[eval_episodes_df["metric"] == metric_name]
        heat = sub.pivot(index="strategy", columns="window", values="F1")

        # Reordenar columnas según las horas
        cols_ordered = [w for w in window_order if w in heat.columns]
        heat = heat[cols_ordered]

        fig_heat = px.imshow(
            heat,
            text_auto=".2f",
            aspect="auto",
            color_continuous_scale="Blues",
            title=f"F1 por estrategia y ventana (episodios) — métrica={metric_name}",
            labels=dict(color="F1")
        )
        fig_heat.update_xaxes(title="Ventana")
        fig_heat.update_yaxes(title="Estrategia")
        fig_heat.show()

    # ============================
    # 3) Barras: F1 por estrategia, ventana y métrica
    # ============================

    fig_bar = px.bar(
        eval_episodes_df,
        x="strategy",
        y="F1",
        color="metric",
        facet_col="window",
        facet_col_wrap=3,
        barmode="group",
        title="F1 de episodios por estrategia, ventana y métrica",
        hover_data=["TP_episodes", "FP_episodes", "FN_episodes"],
        # Ordenar los facet (ventanas) de menor a mayor horas
        category_orders={
            "window": window_order
        }
    )
    fig_bar.update_layout(yaxis=dict(title="F1"), xaxis=dict(title="Estrategia"))
    fig_bar.show()
else:
    print("eval_episodes_df está vacío, no se pueden generar gráficos.")

Métricas por (window, strategy, metric):


,window,strategy,metric,TP_episodes,FP_episodes,FN_episodes,Precision,Recall,F1,coverage_mean,coverage_median,delay_mean_hours,delay_median_hours,false_alarms_per_day
0,12H,decay,ks,21,3,1,0.875000,0.954545,0.913043,0.932722,1.000000,0.950000,0.000000,0.150005
1,12H,decay,mannwhitney,0,14,17,0.000000,0.000000,NaN,0.000000,0.000000,NaN,NaN,0.700024
2,12H,decay,psi,21,2,1,0.913043,0.954545,0.933333,0.926206,1.000000,1.521429,0.000000,0.100003
3,12H,decay,wasserstein,21,1,1,0.954545,0.954545,0.954545,0.926206,1.000000,1.521429,0.000000,0.050002
4,12H,golden,ks,22,24,0,0.478261,1.000000,0.647059,0.943905,1.000000,1.481061,0.000000,1.200042
5,12H,golden,mannwhitney,0,31,17,0.000000,0.000000,NaN,0.000000,0.000000,NaN,NaN,1.550054
6,12H,golden,psi,20,11,2,0.645161,0.909091,0.754717,0.821657,0.990537,2.645833,0.000000,0.550019
7,12H,golden,wasserstein,21,5,1,0.807692,0.954545,0.875000,0.847179,0.990537,3.527778,0.000000,0.250009
8,12H,seasonal,ks,21,21,1,0.500000,0.954545,0.656250,0.933297,1.000000,1.161905,0.000000,1.050036
9,12H,seasonal,mannwhitney,1,34,21,0.028571,0.045455,0.035088,0.005311,0.000000,0.000000,0.000000,1.700059



=== Promedio por ventana y métrica ===



=== Promedio por estrategia y métrica ===



=== Promedio por (métrica, ventana, estrategia) ordenado por F1 ===


,metric,window,strategy,Precision,Recall,F1
9,ks,4H,decay,0.193,1.000,0.324
11,ks,4H,seasonal,0.175,1.000,0.297
10,ks,4H,golden,0.116,1.000,0.208
12,ks,6H,decay,0.368,0.955,0.532
14,ks,6H,seasonal,0.292,0.955,0.447
13,ks,6H,golden,0.184,0.955,0.309
0,ks,12H,decay,0.875,0.955,0.913
2,ks,12H,seasonal,0.500,0.955,0.656
1,ks,12H,golden,0.478,1.000,0.647
3,ks,24H,decay,1.000,0.955,0.977


In [13]:
METRICS_DIR = OUTPUT_DIR / "metrics_drift"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
print(f"📂 Directorio de métricas avanzadas: {METRICS_DIR.resolve()}")

has_cov_mean   = "coverage_mean" in eval_episodes_df.columns
has_delay_mean = "delay_mean_hours" in eval_episodes_df.columns
has_delay_med  = "delay_median_hours" in eval_episodes_df.columns
has_false_rate = "false_alarms_per_day" in eval_episodes_df.columns

# ------------------------------
# 7.1 Calidad en detectar drift
# ------------------------------
quality_cols = ["Precision", "Recall", "F1"]
if has_cov_mean:
    quality_cols.append("coverage_mean")

quality_group = ["metric", "strategy", "window"]

quality_overall = (
    eval_episodes_df
    .groupby(quality_group)[quality_cols]
    .mean()
    .reset_index()
    .sort_values(["metric", "window", "F1"], ascending=[True, True, False])
)
quality_overall.to_csv(METRICS_DIR / "quality_overall.csv", index=False)
print(f"✅ quality_overall.csv guardado ({len(quality_overall)} filas)")

# ------------------------------
# 7.2 Velocidad de detección
# ------------------------------
if has_delay_mean or has_delay_med:
    delay_cols = []
    if has_delay_mean:
        delay_cols.append("delay_mean_hours")
    if has_delay_med:
        delay_cols.append("delay_median_hours")

    speed_overall = (
        eval_episodes_df
        .groupby(quality_group)[delay_cols]
        .mean()
        .reset_index()
        .sort_values(["metric", "window"], ascending=[True, True])
    )
    speed_overall.to_csv(METRICS_DIR / "speed_overall.csv", index=False)
    print(f"✅ speed_overall.csv guardado ({len(speed_overall)} filas)")
else:
    print("ℹ️ No se encontraron columnas de delay en eval_episodes_df → se omiten CSV de velocidad.")

# ------------------------------
# 7.3 Estabilidad (false alarms)
# ------------------------------
if has_false_rate:
    stability_overall = (
        eval_episodes_df
        .groupby(quality_group)[["false_alarms_per_day"]]
        .mean()
        .reset_index()
        .sort_values(["metric", "window", "false_alarms_per_day"],
                     ascending=[True, True, True])
    )
    stability_overall.to_csv(METRICS_DIR / "stability_overall.csv", index=False)
    print(f"✅ stability_overall.csv guardado ({len(stability_overall)} filas)")
else:
    print("ℹ️ No se encontró 'false_alarms_per_day' en eval_episodes_df → se omite stability_overall.csv")

# ------------------------------
# 7.4 Comportamiento por tipo de drift (gradual / abrupto)
#     (usando episodios manuales marcados)
# ------------------------------
has_manual_type = (
    "drift_type" in manual_marked.columns
    and "matched_auto" in manual_marked.columns
    and "coverage" in manual_marked.columns
    and "delay_hours" in manual_marked.columns
)

if has_manual_type and not manual_marked.empty:
    by_type = (
        manual_marked
        .groupby(["metric", "strategy", "window", "drift_type"])
        .agg(
            n_manual=("manual_start", "size"),
            TP=("matched_auto", "sum"),
            Recall=("matched_auto", "mean"),
            coverage_mean=("coverage", "mean"),
            delay_mean_hours=("delay_hours", "mean"),
            delay_median_hours=("delay_hours", "median"),
        )
        .reset_index()
        .sort_values(["metric","drift_type","window"], ascending=[True, True, True])
    )
    by_type.to_csv(METRICS_DIR / "by_drifttype_manual.csv", index=False)
    print(f"✅ by_drifttype_manual.csv guardado ({len(by_type)} filas)")
else:
    print("ℹ️ No se pudo construir análisis por 'drift_type' (faltan columnas o manual_marked está vacío).")

print("🎯 Export de métricas avanzadas terminado.")

📂 Directorio de métricas avanzadas: C:\Users\frncc\OneDrive - Universidad Católica de Chile\Desktop\UC\2025-2\Proyecto de Grado\Proyecto-Grado\Drift Evaluation\synthetic_data\results\metrics_drift
✅ quality_overall.csv guardado (59 filas)
✅ speed_overall.csv guardado (59 filas)
✅ stability_overall.csv guardado (59 filas)
✅ by_drifttype_manual.csv guardado (118 filas)
🎯 Export de métricas avanzadas terminado.


## 8. Visualización 5x2 (export)

In [14]:
def plot_grid_episodes_matplotlib(
    df,
    df_episodes_auto: pd.DataFrame,
    intervals_manual: pd.DataFrame,
    strategy: str,
    window: str,
    metric: str = "psi",
    show_manual: bool = True,
):
    """
    Versión Matplotlib de la grilla 5x2:
      - Línea de la serie.
      - Bandas rojas: episodios manuales.
      - Bandas azules: episodios automáticos (por estrategia, ventana y métrica).
    """
    vars10 = list(df.columns)[:10]
    rows, cols = 5, 2

    subset_auto = df_episodes_auto[
        (df_episodes_auto["strategy"] == strategy) &
        (df_episodes_auto["window"] == window) &
        (df_episodes_auto["metric"] == metric)
    ]

    fig, axes = plt.subplots(rows, cols, figsize=(12, 16), sharex=True)
    axes = axes.flatten()

    for i, var in enumerate(vars10):
        ax = axes[i]
        s = df[var]

        # Serie de tiempo
        ax.plot(s.index, s.values, linewidth=0.8)
        ax.set_title(var, fontsize=9)

        # Episodios manuales (rojo)
        if show_manual:
            mans = intervals_manual[intervals_manual["variable"] == var]
            for _, m in mans.iterrows():
                ax.axvspan(m["manual_start"], m["manual_end"],
                           alpha=0.18, color="red")

        # Episodios automáticos (azul)
        autos = subset_auto[subset_auto["variable"] == var]
        for _, a in autos.iterrows():
            ax.axvspan(a["seg_start"], a["seg_end"],
                       alpha=0.18, color="blue")

        ax.grid(True, alpha=0.3)

    # Si hay menos de 10 variables, apagar ejes sobrantes
    for j in range(len(vars10), len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(
        f"Episodios — strategy={strategy}, window={window}, metric={metric}\n"
        f"(rojo = manual, azul = auto)",
        fontsize=12
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    return fig


In [15]:
from pathlib import Path

EXPORT_ROOT = Path("drift_eval_plots_min_png")
EXPORT_ROOT.mkdir(exist_ok=True)

print("Exportando todas las combinaciones (metric × strategy × window) a PNG (Matplotlib)")

for win in EVAL_WINDOWS:          # ej: ["2H", "4H", "6H", "12H", "24H"]
    # Subcarpeta por ventana
    win_dir = EXPORT_ROOT / f"window_{win}"
    win_dir.mkdir(exist_ok=True)

    for strat in STRATEGIES:      # ["decay", "golden", "seasonal"]
        for metric in METRICS:    # ("psi", "ks", "wasserstein", "mannwhitney")

            subset_auto = df_episodes_auto[
                (df_episodes_auto["strategy"] == strat) &
                (df_episodes_auto["window"]   == win) &
                (df_episodes_auto["metric"]   == metric)
            ]
            if subset_auto.empty:
                # No hay episodios para esta combinación, saltamos
                continue

            print("\n==============================================")
            print(f"Ventana: {win} | Estrategia: {strat} | Métrica: {metric}")
            print("==============================================")

            fig = plot_grid_episodes_matplotlib(
                df=df,
                df_episodes_auto=df_episodes_auto,
                intervals_manual=intervals_manual,
                strategy=strat,
                window=win,
                metric=metric,
                show_manual=True
            )

            out_path = win_dir / f"episodes_{metric}_{strat}_{win}.png"
            fig.savefig(out_path, dpi=200, bbox_inches="tight")
            plt.close(fig)  # importante para no llenar la memoria

            print(f"[OK] Exportado: {out_path}")


Exportando todas las combinaciones (metric × strategy × window) a PNG (Matplotlib)

Ventana: 4H | Estrategia: decay | Métrica: psi
[OK] Exportado: drift_eval_plots_min_png\window_4H\episodes_psi_decay_4H.png

Ventana: 4H | Estrategia: decay | Métrica: ks
[OK] Exportado: drift_eval_plots_min_png\window_4H\episodes_ks_decay_4H.png

Ventana: 4H | Estrategia: decay | Métrica: wasserstein
[OK] Exportado: drift_eval_plots_min_png\window_4H\episodes_wasserstein_decay_4H.png

Ventana: 4H | Estrategia: decay | Métrica: mannwhitney
[OK] Exportado: drift_eval_plots_min_png\window_4H\episodes_mannwhitney_decay_4H.png

Ventana: 4H | Estrategia: golden | Métrica: psi
[OK] Exportado: drift_eval_plots_min_png\window_4H\episodes_psi_golden_4H.png

Ventana: 4H | Estrategia: golden | Métrica: ks
[OK] Exportado: drift_eval_plots_min_png\window_4H\episodes_ks_golden_4H.png

Ventana: 4H | Estrategia: golden | Métrica: wasserstein
[OK] Exportado: drift_eval_plots_min_png\window_4H\episodes_wasserstein_golden